# Train parasite-egg detector (Chula-ParasiteEgg-11) → ONNX
Runtime → Change runtime type → **T4 GPU** (or better).

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')  # checkpoints + final model are saved here

In [ ]:
%cd /content
!git clone https://github.com/rtpsticky/ParasiteEgg.git
%cd ParasiteEgg
!pip -q install -r requirements.txt

## Download from Zenodo (record 15866298, MIT license)

In [ ]:
%%bash
set -e
mkdir -p /content/raw && cd /content/raw
B=https://zenodo.org/records/15866298/files
wget -q --show-progress -c "$B/Chula-ParasiteEgg-11.zip?download=1" -O train.zip
wget -q --show-progress -c "$B/Chula-ParasiteEgg-11_test.zip?download=1" -O test.zip
wget -q -c "$B/test_labels_200.json?download=1" -O test_labels_200.json
echo '1621521969547b0291c0cfbf7e450074  train.zip' | md5sum -c
echo 'a47fdab77ced45bc40b0e8b1e6ff8d22  test.zip' | md5sum -c
unzip -q train.zip -d train && rm train.zip
unzip -q test.zip -d test && rm test.zip
df -h /content

In [ ]:
!python scripts/prepare_dataset.py --src /content/raw/train --out /content/yolo \
   --test-src /content/raw/test --test-json /content/raw/test_labels_200.json --workers 8
!cat /content/yolo/data.yaml

## Train
`yolo11n` is small and fast in the browser. Try `yolo11s.pt` if accuracy isn't enough. After a disconnect, re-run download/prepare, then the resume cell.

In [ ]:
!mkdir -p /content/drive/MyDrive/parasite_runs && ln -sfn /content/drive/MyDrive/parasite_runs runs
!python scripts/train.py --data /content/yolo/data.yaml --model yolo11n.pt --epochs 100 --device 0

In [ ]:
# resume after a disconnect
# !python scripts/train.py --resume

## Evaluate on test + export ONNX

In [ ]:
!python scripts/export_onnx.py --weights runs/detect/parasite/weights/best.pt --data /content/yolo/data.yaml --out /content/drive/MyDrive/parasite_runs/web_model
!ls -la /content/drive/MyDrive/parasite_runs/web_model